## Fine tuning flan-t5-base model using LoRA technique for dialogue summarization dataset

In this notebook, we will fine-tune an existing LLM from Hugging Face for enhanced dialogue summarization. we will use the [FLAN-T5](https://huggingface.co/docs/transformers/model_doc/flan-t5) model, which provides a high quality instruction tuned model and can summarize text out of the box. To improve the inferences, you will explore a full fine-tuning approach and evaluate the results with ROUGE metrics. Then you will perform Parameter Efficient Fine-Tuning (PEFT), evaluate the resulting model and see that the benefits of PEFT outweigh the slightly-lower performance metrics.

### Loading libraries

In [1]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
import evaluate
import pandas as pd
import numpy as np

/home/utsajinlab/anaconda3/envs/fine_tuning/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



### Load Dataset and LLM

You are going to continue experimenting with the [DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) Hugging Face dataset. It contains 10,000+ dialogues with the corresponding manually labeled summaries and topics. 

In [2]:
huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [3]:
dataset['train']['dialogue'][1]

"#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little."

### Loading the model

In [4]:
model_name='google/flan-t5-large'

original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [5]:
def model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f"trainable model parameters: {trainable_model_params}"

print(model_parameters(original_model))

trainable model parameters: 783150080


It is possible to pull out the number of model parameters and find out how many of them are trainable. The following function can be used to do that, at this stage, you do not need to go into details of it. 

### Test the Model before fine-tuning


In [6]:
# index = 200

# dialogue = dataset['test'][index]['dialogue']
# summary = dataset['test'][index]['summary']

# prompt = f"""
# Summarize the following conversation.

# {dialogue}

# Summary:
# """

# inputs = tokenizer(prompt, return_tensors='pt')
# output = tokenizer.decode(
#     original_model.generate(
#         inputs["input_ids"], 
#         max_new_tokens=200,
#     )[0], 
#     skip_special_tokens=True
# )

# print(f'Input prompt:{prompt}')
# print(f'\n\n Human summary: \n{summary}\n')
# print(f'\n\nOriginal model summarization before fine-tuning: \n\n{output}')

In [7]:
def tokenize_function(example):
    start_prompt = 'Summarize the following conversation.\n\n'
    end_prompt = '\n\nSummary: '
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example["dialogue"]]
    example['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt",max_length=512).input_ids
    example['labels'] = tokenizer(example["summary"], padding="max_length", truncation=True, return_tensors="pt", max_length=128).input_ids
    
    return example

# The dataset actually contains 3 diff splits: train, validation, test.
# The tokenize_function code is handling all data across all splits in batches.
tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['id', 'topic', 'dialogue', 'summary',])

Map: 100%|███████████████████████████| 500/500 [00:00<00:00, 3564.20 examples/s]


In [8]:
print(tokenized_datasets)
print(tokenized_datasets['train'][0])


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 500
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1500
    })
})
{'input_ids': [12198, 1635, 1737, 8, 826, 3634, 5, 1713, 345, 13515, 536, 4663, 10, 2018, 6, 1363, 5, 3931, 5, 27, 31, 51, 7582, 12833, 77, 7, 5, 1615, 33, 25, 270, 469, 58, 1713, 345, 13515, 357, 4663, 10, 27, 435, 34, 133, 36, 3, 9, 207, 800, 12, 129, 3, 9, 691, 18, 413, 5, 1713, 345, 13515, 536, 4663, 10, 2163, 6, 168, 6, 25, 43, 29, 31, 17, 141, 80, 21, 305, 203, 5, 148, 225, 43, 80, 334, 215, 5, 1713, 345, 13515, 357, 4663, 10, 27, 214, 5, 27, 2320, 38, 307, 38, 132, 19, 1327, 1786, 6, 572, 281, 217, 8, 2472, 58, 1713, 345, 13515, 536, 4663, 10, 1548, 6, 8, 200, 194, 12, 1792, 2261, 21154, 19, 12, 253, 91, 81, 135, 778, 5, 264, 653, 12, 369, 44, 709, 728, 3, 9, 215, 21, 39, 293, 207,

In [9]:
print(tokenizer.decode(tokenized_datasets['train'][0]['input_ids'],  skip_special_tokens=True))
print(tokenizer.decode(tokenized_datasets['train'][0]['labels'],  skip_special_tokens=True))

Summarize the following conversation. #Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today? #Person2#: I found it would be a good idea to get a check-up. #Person1#: Yes, well, you haven't had one for 5 years. You should have one every year. #Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor? #Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good. #Person2#: Ok. #Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith? #Person2#: Yes. #Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit. #Person2#: I've tried hundreds of times, but I just can't seem to kick the habit. #Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave. #Person2#: Ok, thanks doctor. Summary: 
Mr. Smith's getting 

In [10]:
truncated_inputs = sum([len(tokenizer.encode(text)) > 512 for text in dataset['train']['dialogue']])
truncated_outputs = sum([len(tokenizer.encode(text)) > 128 for text in dataset['train']['summary']])
print(f"Truncated Inputs: {truncated_inputs}, Truncated Outputs: {truncated_outputs}")


Token indices sequence length is longer than the specified maximum sequence length for this model (517 > 512). Running this sequence through the model will result in indexing errors


Truncated Inputs: 238, Truncated Outputs: 21



## Perform Parameter Efficient Fine-Tuning (PEFT)

Now, let's perform **Parameter Efficient Fine-Tuning (PEFT)** fine-tuning as opposed to "full fine-tuning" as you did above. PEFT is a form of instruction fine-tuning that is much more efficient than full fine-tuning - with comparable evaluation results as you will see soon. 

PEFT is a generic term that includes **Low-Rank Adaptation (LoRA)** and prompt tuning (which is NOT THE SAME as prompt engineering!). In most cases, when someone says PEFT, they typically mean LoRA. LoRA, at a very high level, allows the user to fine-tune their model using fewer compute resources (in some cases, a single GPU). After fine-tuning for a specific task, use case, or tenant with LoRA, the result is that the original LLM remains unchanged and a newly-trained “LoRA adapter” emerges. This LoRA adapter is much, much smaller than the original LLM - on the order of a single-digit % of the original LLM size (MBs vs GBs).  

That said, at inference time, the LoRA adapter needs to be reunited and combined with its original LLM to serve the inference request.  The benefit, however, is that many LoRA adapters can re-use the original LLM which reduces overall memory requirements when serving multiple tasks and use cases.


###  Setup the PEFT/LoRA model for Fine-Tuning

You need to set up the PEFT/LoRA model for fine-tuning with a new layer/parameter adapter. Using PEFT/LoRA, you are freezing the underlying LLM and only training the adapter. Have a look at the LoRA configuration below. Note the rank (`r`) hyper-parameter, which defines the rank/dimension of the adapter to be trained.

In [11]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=64, # Rank
    lora_alpha=64,
    target_modules=["q", "v"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM # FLAN-T5
)

Add LoRA adapter layers/parameters to the original LLM to be trained.

In [12]:
peft_model = get_peft_model(original_model, 
                            lora_config)
print(model_parameters(peft_model))

trainable model parameters: 18874368


<a name='3.2'></a>
### 3.2 - Train PEFT Adapter

Define training arguments and create `Trainer` instance.

In [13]:
output_dir = "Fine_tuning_FlanT5_large_model_LoRA_summerization_test_512_128"


peft_training_args = TrainingArguments(
    output_dir=f'{output_dir}/model',
    logging_dir=f'{output_dir}/logs',

    auto_find_batch_size=True,
    learning_rate=1e-4, # Higher learning rate than full fine-tuning.
    lr_scheduler_type="cosine",    
    num_train_epochs=40,
    warmup_steps=200,
    #label_smoothing_factor=0.1,
    weight_decay=0.01,
    #logging_steps=1,
    #max_steps=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",


    # Evaluation & Logging
    evaluation_strategy="steps",
    eval_steps=1000,  
    save_steps=1000,
    logging_strategy="steps",
    logging_steps=10,  
    save_total_limit=2,
)
    
peft_trainer = Trainer(
    model=peft_model,
    args=peft_training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"]
)

/home/utsajinlab/anaconda3/envs/fine_tuning/lib/python3.9/site-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [14]:
peft_trainer.train()

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.
/home/utsajinlab/anaconda3/envs/fine_tuning/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss,Validation Loss
1000,0.353500,0.290875
2000,0.317100,0.277879
3000,0.320300,0.273217
4000,0.278400,0.270490
5000,0.293200,0.269533
6000,0.308600,0.270598
7000,0.282400,0.267883
8000,0.248100,0.271119
9000,0.282600,0.272561
10000,0.268000,0.271484


/home/utsajinlab/anaconda3/envs/fine_tuning/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/utsajinlab/anaconda3/envs/fine_tuning/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/utsajinlab/anaconda3/envs/fine_tuning/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/home/utsajinlab/anaconda3/envs/fine_tuning/lib/python3.9/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return

TrainOutput(global_step=15560, training_loss=0.46467680158835756, metrics={'train_runtime': 55430.4979, 'train_samples_per_second': 8.991, 'train_steps_per_second': 0.281, 'total_flos': 1.1746175396452762e+18, 'train_loss': 0.46467680158835756, 'epoch': 39.89887640449438})

In [16]:
# Save the model
save_dir = f'{output_dir}/final'

peft_trainer.save_model(save_dir)

tokenizer.save_pretrained(save_dir)

print(f"Final model saved to: {save_dir}")

Final model saved to: Fine_tuning_FlanT5_large_model_LoRA_summerization_test/final


In [13]:
# peft_trainer.train()

# peft_model_path="./peft-dialogue-summary-checkpoint-local"

# peft_trainer.model.save_pretrained(peft_model_path)
# tokenizer.save_pretrained(peft_model_path)

Prepare this model by adding an adapter to the original FLAN-T5 model. You are setting `is_trainable=False` because the plan is only to perform inference with this PEFT model. If you were preparing the model for further training, you would set `is_trainable=True`.

In [17]:

model_name='google/flan-t5-large'

original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [18]:
from peft import PeftModel, PeftConfig
save_dir = f'{output_dir}/final'

peft_model_base = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

#instruct_model = AutoModelForSeq2SeqLM.from_pretrained(save_dir, local_files_only=True)
peft_model = PeftModel.from_pretrained(peft_model_base, 
                                       save_dir, 
                                       torch_dtype=torch.bfloat16,
                                       is_trainable=False)
peft_model = peft_model.merge_and_unload()


In [19]:
# Check if LoRA adapters are applied
print(f"Active adapters: {peft_model.active_adapters}")


Active adapters: <bound method PeftAdapterMixin.active_adapters of T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_

<a name='3.3'></a>
### 3.3 - Evaluate the Model Qualitatively (Human Evaluation)

Make inferences for the same example as in sections [1.3](#1.3) and [2.3](#2.3), with the original model, fully fine-tuned and PEFT model.

In [20]:
def inference(text, model, tokenizer, max_input_tokens=512, max_output_tokens=128):
    # Tokenize input
    input_ids = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_tokens
    )["input_ids"].to(model.device)

    # Generate response
    with torch.no_grad():
        generated_tokens = model.generate(
            input_ids=input_ids,
            max_length=max_output_tokens,  
            pad_token_id=tokenizer.eos_token_id  
        )

    # Decode and strip input prompt
    generated_text = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
    

    return generated_text


In [21]:
index = 1
dialogue = dataset['test'][index]['dialogue']
human_baseline_summary = dataset['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""


original_model_text_output = inference(dialogue,original_model, tokenizer )
peft_model_text_output = inference(dialogue,peft_model, tokenizer )

print(f'BASELINE HUMAN SUMMARY:\n{human_baseline_summary}')
print(f'ORIGINAL MODEL:\n{original_model_text_output}')
print(f'PEFT MODEL:\n{peft_model_text_output}')

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BASELINE HUMAN SUMMARY:
In order to prevent employees from wasting time on Instant Message programs, #Person1# decides to terminate the use of those programs and asks Ms. Dawson to send out a memo to all employees by the afternoon.
ORIGINAL MODEL:
#Person1: Ms. Dawson, please take dictation for me.
PEFT MODEL:
#Person1# asks Ms. Dawson to take a dictation for #Person1#. #Person1# tells Ms. Dawson that all office communications are restricted to email correspondence and official memos. The use of Instant Message programs by employees during working hours is strictly prohibited. Ms. Dawson thinks it wastes too much time. #Person1# says any employee who persists in using Instant Messaging will receive a warning and be placed on probation.



### Evaluate the Model Quantitatively (with ROUGE Metric)

The [ROUGE metric](https://en.wikipedia.org/wiki/ROUGE_(metric)) helps quantify the validity of summarizations produced by models. It compares summarizations to a "baseline" summary which is usually created by a human. While not perfect, it does indicate the overall increase in summarization effectiveness that we have accomplished by fine-tuning.

In [22]:
rouge = evaluate.load('rouge')

Generate the outputs for the sample of the test dataset (only 10 dialogues and summaries to save time), and save the results.

In [23]:
dialogues = dataset['test'][1100:1200]['dialogue']
human_baseline_summaries = dataset['test'][1100:1200]['summary']

original_model_summaries = []
peft_model_summaries = []

for idx, dialogue in enumerate(dialogues):
    prompt = f"""
Summarize the following conversation.

{dialogue}

Summary: """
    
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids

    human_baseline_text_output = human_baseline_summaries[idx]
    
    original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
    original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)

    peft_model_outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200))
    peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

    original_model_summaries.append(original_model_text_output)
    peft_model_summaries.append(peft_model_text_output)

zipped_summaries = list(zip(human_baseline_summaries, original_model_summaries, peft_model_summaries))
 
df = pd.DataFrame(zipped_summaries, columns = ['human_baseline_summaries', 'original_model_summaries', 'peft_model_summaries'])
df

Token indices sequence length is longer than the specified maximum sequence length for this model (559 > 512). Running this sequence through the model will result in indexing errors


,human_baseline_summaries,original_model_summaries,peft_model_summaries
0,#Person2# asks #Person1# about a new service a...,#Person2# received leaflets in the post from t...,#Person2# received some leaflets in the post f...
1,#Person1# would like to store his luggage in #...,#Person1 is leaving in 30 minutes. He needs a ...,#Person1# is checking out and wants to put his...
2,"#Person1# wants to hit some place, but he hesi...",#Person1 is leaving in 30 minutes. He needs a ...,#Person1# is checking out and wants to put his...
3,#Person1# wants to hit some place. So he needs...,#Person1 is leaving in 30 minutes. He needs a ...,#Person1# is checking out and wants to put his...
4,#Person1# wants to know the charge at #Person2...,The party is for three people. They are going ...,#Person2# is having a party. #Person1# tells #...
...,...,...,...
95,#Person2# asks his daughter about admission re...,#Person2# is considering quitting her job and ...,#Person1#'s dad asks #Person1# about the admis...
96,#Person2# wants to know admission requirement ...,#Person2# is considering quitting her job and ...,#Person1#'s dad asks #Person1# about the admis...
97,Kalina calls Professor Clark that she needs to...,Kalina will miss a few days of school because ...,Kalina calls Professor Clark to tell him she w...
98,Kalina calls Professor Clark and asks for leav...,Kalina will miss a few days of school because ...,Kalina calls Professor Clark to tell him she w...


In [28]:
print(df.iloc[3,0])

print(df.iloc[3,1])

#Person1# wants to hit some place. So he needs leaving the luggage in #Person2#'s place, but there is a deposit. He only has 30 minutes to consider.
#Person1 is leaving in 30 minutes. He needs a place to put his luggage. #Person2 offers a storage space for a small charge, plus a deposit. #Person1 will present his VISA to cover the deposit.


#### Compute ROUGE score for this subset of the data. 

In [24]:
rouge = evaluate.load('rouge')

original_model_results = rouge.compute(
    predictions=original_model_summaries,
    references=human_baseline_summaries[0:len(original_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

peft_model_results = rouge.compute(
    predictions=peft_model_summaries,
    references=human_baseline_summaries[0:len(peft_model_summaries)],
    use_aggregator=True,
    use_stemmer=True,
)

print('ORIGINAL MODEL:')
print(original_model_results)

print('PEFT MODEL:')
print(peft_model_results)

ORIGINAL MODEL:
{'rouge1': np.float64(0.29122621899486345), 'rouge2': np.float64(0.08034173563946087), 'rougeL': np.float64(0.22756085948196977), 'rougeLsum': np.float64(0.22736029707730482)}
PEFT MODEL:
{'rouge1': np.float64(0.4208838300693395), 'rouge2': np.float64(0.1519732828066001), 'rougeL': np.float64(0.32728226803081695), 'rougeLsum': np.float64(0.3274397840981834)}
